In [ ]:
import os, tqdm 
import requests
from bs4 import BeautifulSoup
import hashlib
import re

In [ ]:
# === Configuration ===
SERPER_API_KEY = os.getenv("SERPER_API_KEY")  # set in your environment
SERPER_API_URL = "https://google.serper.dev/search"

In [ ]:
def hash_query(query: str) -> str:
    return hashlib.md5(query.encode("utf-8")).hexdigest()

In [ ]:
def search_mayo_clinic_top1_serper(query):
    headers = {"X-API-KEY": SERPER_API_KEY, "Content-Type": "application/json"}
    query_with_site = f"{query} site:mayoclinic.org"
    payload = {"q": query_with_site}

    response = requests.post(SERPER_API_URL, headers=headers, json=payload)
    if response.status_code != 200:
        raise Exception(f"Serper API error: {response.text}")

    data = response.json()
    mayo_urls = [
        item["link"] for item in data.get("organic", [])
        if "mayoclinic.org" in item["link"]
    ]

    if not mayo_urls:
        print("❌ No Mayo Clinic URL found in search results.")
        return None
    return mayo_urls[0]


In [ ]:
def scrape_mayo_page_text(url):
    try:
        response = requests.get(url, timeout=10)
        soup = BeautifulSoup(response.content, "html.parser")
        paragraphs = soup.find_all("p")
        clean_text = "\n".join(p.get_text() for p in paragraphs if len(p.get_text()) > 40)
        return clean_text.strip()
    except Exception as e:
        print(f"⚠️ Error scraping {url}: {e}")
        return ""

In [ ]:
def get_reference_knowledge_from_conversation_serper(conversation_path, save_path):
    with open(conversation_path, "r") as f:
        conv_text = f.read()

    # Extract the first User question as the query
    user_lines = re.findall(r"User: (.+)", conv_text)
    if not user_lines:
        print("⚠️ No user prompt found.")
        return ""
    query = user_lines[0]
    print(f"\n🔍 Searching Mayo Clinic for query:\n{query}\n")

    url = search_mayo_clinic_top1_serper(query)
    if not url:
        print("❌ Failed to find Mayo Clinic URL.")
        return ""

    print(f"🌐 Mayo URL: {url}")

    page_text = scrape_mayo_page_text(url)
    if not page_text:
        print("⚠️ Failed to extract page content.")
        return ""

    # save
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    with open(save_path, "w") as f:
        f.write(page_text)

    print(f"\n📄 Saved Mayo Clinic reference to: {save_path}")
    return page_text

In [ ]:
CONVERSATION_DIR = "outputs/data/conversations_short"
REFERENCE_DIR = "outputs/data/evidence/mayo"
os.makedirs(REFERENCE_DIR, exist_ok=True)

# Get all conversation files
conversation_files = sorted([
    f for f in os.listdir(CONVERSATION_DIR)
    if f.startswith("conversation_") and f.endswith(".txt")
])

# batch processing
for filename in tqdm(conversation_files, desc="Fetching Mayo references"):
    idx = filename.split("_")[1].split(".")[0]
    conversation_path = os.path.join(CONVERSATION_DIR, filename)
    reference_path = os.path.join(REFERENCE_DIR, f"evidence_{idx}.txt")

    if os.path.exists(reference_path):
        print(f"✅ Evidence already exists for conversation {idx}, skipping.")
        continue

    try:
        get_reference_knowledge_from_conversation_serper(
            conversation_path,
            reference_path
        )
    except Exception as e:
        print(f"❌ Error for conversation {idx}: {e}")
